In [1]:
# Imports and setup
import json
import warnings
import torch
from pprint import pprint

import numpy as np
import pandas as pd

from benchmark.config import DATA_PROCESSED, N_TRIALS, NUM_FOLDS
from benchmark.models.xgboost_model import (
    build_pipeline_reg,
    build_preprocessor,
    evaluate_reg,
    export_results_reg,
    final_fit_reg,
    get_search_space,
    run_search_reg,
)

warnings.filterwarnings("ignore")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Main config
data_name = "abalone"
processed = DATA_PROCESSED / data_name
k_folds = NUM_FOLDS
n_trials = N_TRIALS

xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
# xgb_device = "cpu"
print("XGBoost device:", xgb_device)

XGBoost device: cuda


In [3]:
# Load data and metadata
train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as file:
    metadata = json.load(file)

print("Dataset path:", processed)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nMetadata:")
pprint(metadata)

Dataset path: /home/shado/college_project/comp9417/9417GroupProject/data/processed/abalone
Train shape: (3341, 9)
Test shape: (836, 9)

Metadata:
{'cat_cols': ['sex'],
 'num_cols': ['length',
              'diameter',
              'height',
              'whole_weight',
              'shucked_weight',
              'viscera_weight',
              'shell_weight'],
 'target_col': 'rings'}


In [4]:
# Split x and y
target_column = metadata["target_col"]

x_train_full = train_df.drop(columns=[target_column]).copy()
y_train_full = train_df[target_column].to_numpy(dtype=np.float32)

x_test = test_df.drop(columns=[target_column]).copy()
y_test = test_df[target_column].to_numpy(dtype=np.float32)

In [5]:
# Build preprocessor and run hyperparameter search
preprocessor = build_preprocessor(metadata)
pipeline = build_pipeline_reg(preprocessor, device=xgb_device, n_jobs=1)
search_space = get_search_space()

best_search_params, best_model_params, best_cv_mse, search_time = run_search_reg(
    pipeline,
    search_space,
    x_train_full,
    y_train_full,
    k_folds=k_folds,
    n_trials=n_trials,
)

print("\nBest CV MSE:", best_cv_mse)
print("\nBest hyperparameters:")
pprint(best_model_params)


Best CV MSE: 4.526727962493896

Best hyperparameters:
{'colsample_bytree': 0.8918729886843237,
 'gamma': 2.0078371758085833e-05,
 'learning_rate': 0.01382779021382068,
 'max_depth': 5,
 'min_child_weight': 7,
 'n_estimators': 422,
 'reg_alpha': 1.924588157370848e-07,
 'reg_lambda': 8.473004754265682,
 'subsample': 0.6506169490401995}


In [6]:
# Final fit with best params
final_pipeline, fit_time = final_fit_reg(
    preprocessor,
    best_search_params,
    x_train_full,
    y_train_full,
    device=xgb_device,
)

In [7]:
# Evaluate on train and test
train_metrics = evaluate_reg(final_pipeline, x_train_full, y_train_full)
test_metrics = evaluate_reg(final_pipeline, x_test, y_test)

# %%
# Final results
print("\nFinal results:")

print("Search time (seconds):", search_time)
print("Final training time (seconds):", fit_time)
print("Best CV MSE:", best_cv_mse)

print("\nTrain RMSE:", train_metrics["rmse"])
print(
    "Train inference time per sample (seconds):",
    train_metrics["infer_time_per_sample_s"],
)

print("\nTest RMSE:", test_metrics["rmse"])
print(
    "Test inference time per sample (seconds):", test_metrics["infer_time_per_sample_s"]
)


Final results:
Search time (seconds): 32.561225216999446
Final training time (seconds): 0.18946694400074193
Best CV MSE: 4.526727962493896

Train RMSE: 1.8218427782824476
Train inference time per sample (seconds): 6.196725831030269e-06

Test RMSE: 2.1737427947510106
Test inference time per sample (seconds): 2.68839832651401e-06


In [8]:
# Export model and results
export_results_reg(
    data_name,
    final_pipeline,
    best_model_params,
    train_metrics,
    test_metrics,
    fit_time,
    search_time,
    best_cv_mse,
)


Model saved to /home/shado/college_project/comp9417/9417GroupProject/models/abalone/xgboost.joblib
Pipeline saved to /home/shado/college_project/comp9417/9417GroupProject/models/abalone/xgboost_pipeline.joblib
Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/abalone/xgboost.json
